In [4]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = r"C:\git\CleverCheck\server\my_model\my_trained_dictabert"

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)

model.eval()



def predict(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=256
    )

    with torch.no_grad():
        logits = model(**inputs).logits

    score = logits.view(-1)[0].item()

    return score



In [31]:
import requests

session = requests.Session()

# 1. פותחים את האתר כדי לקבל cookies
session.get("https://www.scribens.com", verify=False)

# 2. שולחים request אמיתי
url = "https://www.scribens.fr/Scribens/OtherAlg_Ref_Servlet"

data = {
    "FunctionName": "Get_Correction",
    "Plugin": "Website_desktop",
    "Text": "השמש וכוכבי הלחת@ שמסובבים אותהח",
    "IdLanguage": "he",
    "IdLangDisplay": "he",
    "Tone": "nope",
    "Settings": "points:none|title:no|conclusion:no|inclusive:no|function:None"
}

headers = {
    "User-Agent": "Mozilla/5.0",
    "Origin": "https://www.scribens.com",
    "Referer": "https://www.scribens.com/",
}

res = session.post(url, data=data, headers=headers, verify=False)

print(res.text)

{"Map_Solutions":{},"ResultSt":"השמש וכוכבי הלכת שמסובבים אותה","NbRewriting":-1,"NbSummarizing":-1,"NbTranslation":-1,"DisplayPremiumPanel":false}


In [8]:
import pandas as pd
import re
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =========================
# מודל
# =========================
model_path = r"C:\git\CleverCheck\server\my_model\my_trained_dictabert"

tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


# =========================
# פונקציית בניית טקסט
# =========================
def format_input(q, t, s):
    return f"[Q]{q}[T]{t}[S]{s}"


# =========================
# חילוץ שדות מתוך טקסט
# =========================
def extract_q_t_s(text):
    if not isinstance(text, str):
        return "", "", ""

    q = re.search(r"\[Q\]\s*(.*?)\s*\[T\]", text)
    t = re.search(r"\[T\]\s*(.*?)\s*\[S\]", text)
    s = re.search(r"\[S\]\s*(.*)", text)

    return (
        q.group(1).strip() if q else "",
        t.group(1).strip() if t else "",
        s.group(1).strip() if s else ""
    )


# =========================
# קריאת קובץ
# =========================
file_path = r"Y:\\שונות\\מלכי וציפי\\test_with_model_score_new_new.csv"

df = pd.read_csv(file_path, encoding="utf-8-sig", engine="python")

# חילוץ q,t,s
df[["q", "t", "s"]] = df["text"].apply(lambda x: pd.Series(extract_q_t_s(x)))

# סינון שורות ריקות
df = df[(df["q"] != "") & (df["t"] != "") & (df["s"] != "")].reset_index(drop=True)


# =========================
# יצירת טקסטים למודל
# =========================
texts = [format_input(r.q, r.t, r.s) for r in df.itertuples(index=False)]


# =========================
# אינפרנס בבאצ'ים
# =========================
BATCH_SIZE = 32
results = []

with torch.no_grad():
    for i in range(0, len(texts), BATCH_SIZE):
        batch_texts = texts[i:i + BATCH_SIZE]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=256
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        logits = model(**inputs).logits

        # התאמה לבינארי / מולטי-קלאס
        if logits.shape[1] == 1:
            scores = torch.sigmoid(logits).squeeze(-1)
        else:
            scores = torch.softmax(logits, dim=1)[:, 1]

        results.extend(scores.cpu().tolist())


# =========================
# הוספה ושמירה
# =========================
df["dictabert_score"] = results

output_path = file_path.replace(".csv", "_with_dictabert.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("נשמר:", output_path)

נשמר: Y:\\שונות\\מלכי וציפי\\test_with_model_score_new_new_with_dictabert.csv


In [6]:
import os
print(os.getcwd())

C:\git\CleverCheck\server\services
